In [88]:
# Install required libraries

!pip install -q streamlit google-genai pyngrok

print("Libraries installed successfully!")

Libraries installed successfully!


In [89]:
import os

project_folder = "AI_Escape_Room"

folders = [
    "version_1",
    "version_2",
    "screenshots",
    "documentation"
]

os.makedirs(project_folder, exist_ok=True)

for folder in folders:
    os.makedirs(
        os.path.join(project_folder, folder),
        exist_ok=True
    )

print("Project folders created successfully!")

print(os.listdir(project_folder))

Project folders created successfully!
['documentation', 'version_2', 'screenshots', 'version_1']


In [108]:
%%writefile /content/AI_Escape_Room/version_1/app.py

import streamlit as st
from google import genai

# -----------------------------
# PAGE CONFIGURATION
# -----------------------------

st.set_page_config(
    page_title="AI Escape Room",
    page_icon="🔐",
    layout="centered"
)

# -----------------------------
# GAME INFORMATION
# -----------------------------

GAME_TITLE = "The Mystery of the Locked Library"

ROOM_OBJECTS = [
    "Desk",
    "Bookshelf",
    "Painting",
    "Locked Drawer",
    "Exit Door"
]

SYSTEM_PROMPT = """
You are the Game Master of an interactive AI Escape Room.

STORY:
The player wakes up inside a mysterious locked library.

ROOM OBJECTS:
1. Desk
2. Bookshelf
3. Painting
4. Locked Drawer
5. Exit Door

PUZZLES:

Puzzle 1:
The bookshelf contains four special books.
Their numbers are 1, 2, 3 and 4.
The player must discover the sequence 1234.

Puzzle 2:
The painting contains a hidden message:
LOOK BEHIND THE PAINTING.
The player discovers a small key.

Puzzle 3:
The desk contains a note explaining that the
final exit requires the four-digit code 2468.

FINAL CHALLENGE:
The player must discover the clues and enter
2468 to escape.

RULES:
    - Act as an intelligent and helpful AI Game Master.
    - Respond naturally and clearly to player messages.
    - Understand the player's questions before responding.
    - Give specific clues related to the actual puzzles.
    - Never give vague or repetitive descriptions.
    - When players inspect an object, explain its relevant clues.
    - When players ask for exact numbers, provide the
      numbers available in the puzzle information.
    - When players request hints, provide useful hints
      without unnecessarily revealing every answer.
    - If players explicitly request a puzzle solution,
      provide the correct answer.
    - Never invent puzzle numbers, clues or objects.
    - Maintain conversation context.
    - Explain incorrect answers politely.
    - Do not claim the player escaped unless the
      required final code has been entered.
    - Keep the game engaging and interactive.
"""

# -----------------------------
# SESSION STATE
# -----------------------------

if "messages" not in st.session_state:

    st.session_state.messages = [
        {
            "role": "assistant",
            "content": """
Welcome to the Mystery of the Locked Library!

You wake up inside an unfamiliar library.
The exit door is locked.

You notice five objects:

📚 Bookshelf
🖼️ Painting
🪑 Desk
🔒 Locked Drawer
🚪 Exit Door

Explore the room and discover clues to escape!

What would you like to investigate first?
"""
        }
    ]

if "chat" not in st.session_state:
    st.session_state.chat = None

# -----------------------------
# SIDEBAR
# -----------------------------

st.sidebar.title("🔐 Game Information")

st.sidebar.write("### Available Objects")

for obj in ROOM_OBJECTS:
    st.sidebar.write("•", obj)

st.sidebar.info(
    "Your mission is to investigate the "
    "library and escape."
)

api_key = st.sidebar.text_input(
    "Gemini API Key",
    type="password",
    help="Your API key is used only for this session."
)

if st.sidebar.button("Restart Game"):
    st.session_state.messages = [
        {
            "role": "assistant",
            "content": "Welcome back! You are inside the locked library. What would you like to investigate?"
        }
    ]
    st.session_state.chat = None
    st.rerun()

# -----------------------------
# MAIN INTERFACE
# -----------------------------

st.title("🔐 AI Escape Room")

st.subheader(GAME_TITLE)

st.write(
    "Explore the mysterious library, solve puzzles "
    "and find your way out!"
)

st.divider()

# Display conversation history

for message in st.session_state.messages:

    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# -----------------------------
# PLAYER INTERACTION
# -----------------------------

player_input = st.chat_input(
    "What would you like to do?"
)

if player_input:

    st.session_state.messages.append(
        {
            "role": "user",
            "content": player_input
        }
    )

    with st.chat_message("user"):
        st.markdown(player_input)

    with st.chat_message("assistant"):

        if not api_key:

            response_text = (
                "Please enter your Gemini API key "
                "in the sidebar to start playing."
            )

        else:

            try:

                client = genai.Client(
                    api_key=api_key
                )

                conversation = SYSTEM_PROMPT + "\n\n"

                for msg in st.session_state.messages:
                    conversation += (
                        msg["role"].upper()
                        + ": "
                        + msg["content"]
                        + "\n"
                    )

                response = client.models.generate_content(
                    model="gemini-3.8-flash",
                    contents=conversation
                )

                response_text = response.text

            except Exception as e:

                response_text = (
                    "Something went wrong while "
                    "connecting to Gemini. Please "
                    "check your API key and try again.\n\n"
                    f"Error: {e}"
                )

        st.markdown(response_text)

    st.session_state.messages.append(
        {
            "role": "assistant",
            "content": response_text
        }
    )

Overwriting /content/AI_Escape_Room/version_1/app.py


In [91]:
!streamlit run /content/AI_Escape_Room/version_1/app.py \
    --server.port 8501 \
    --server.address 0.0.0.0 \
    > /content/streamlit.log 2>&1 &

In [92]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [93]:
!tail -30 /content/streamlit.log



2026-11-14 16:19:12.930 Port 8501 is not available
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [94]:
from google import genai
from getpass import getpass

# Enter API key securely
api_key = getpass("Enter your Gemini API key: ")

# Connect to Gemini
client = genai.Client(api_key=api_key)

# Test AI response
try:
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents="Reply with: Gemini connection successful!"
    )

    print("SUCCESS!")
    print(response.text)

except Exception as e:
    print("CONNECTION ERROR:")
    print(type(e).__name__, str(e))

Enter your Gemini API key: ··········
CONNECTION ERROR:
ServerError 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


In [95]:
from google import genai
from getpass import getpass

api_key = getpass("Enter Gemini API key: ")

client = genai.Client(api_key=api_key)

try:
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents="Say: AI Escape Room connection successful!"
    )

    print(response.text)

except Exception as e:
    print("Error:", e)

Enter Gemini API key: ··········
AI Escape Room connection successful!


In [96]:
from pathlib import Path

app_path = Path("/content/AI_Escape_Room/version_1/app.py")

code = app_path.read_text()

code = code.replace(
    "gemini-3.8-flash",
    "gemini-3.5-flash-lite"
)

app_path.write_text(code)

print("Version 1 updated successfully!")

Version 1 updated successfully!


Version 2

In [97]:
import shutil
import os

source = "/content/AI_Escape_Room/version_1/app.py"
destination = "/content/AI_Escape_Room/version_2/app.py"

shutil.copy(source, destination)

print("Version 2 created successfully!")
print("Version 1 preserved safely!")

Version 2 created successfully!
Version 1 preserved safely!


In [98]:
import os

v2_path = "/content/AI_Escape_Room/version_2/app.py"

print("Version 2 file exists:", os.path.exists(v2_path))
print("Version 2 is ready for improvements!")

Version 2 file exists: True
Version 2 is ready for improvements!


In [99]:
v2_path = "/content/AI_Escape_Room/version_2/app.py"

with open(v2_path, "r", encoding="utf-8") as file:
    code = file.read()

print(code)


import streamlit as st
from google import genai

# -----------------------------
# PAGE CONFIGURATION
# -----------------------------

st.set_page_config(
    page_title="AI Escape Room",
    page_icon="🔐",
    layout="centered"
)

# -----------------------------
# GAME INFORMATION
# -----------------------------

GAME_TITLE = "The Mystery of the Locked Library"

ROOM_OBJECTS = [
    "Desk",
    "Bookshelf",
    "Painting",
    "Locked Drawer",
    "Exit Door"
]

SYSTEM_PROMPT = """
You are the Game Master of an interactive AI Escape Room.

STORY:
The player wakes up inside a mysterious locked library.

ROOM OBJECTS:
1. Desk
2. Bookshelf
3. Painting
4. Locked Drawer
5. Exit Door

PUZZLES:

Puzzle 1:
The bookshelf contains four special books.
Their numbers are 1, 2, 3 and 4.
The player must discover the sequence 1234.

Puzzle 2:
The painting contains a hidden message:
LOOK BEHIND THE PAINTING.
The player discovers a small key.

Puzzle 3:
The desk contains a note explaining that the
fin

In [100]:
import os

v2_path = "/content/AI_Escape_Room/version_2/app.py"

print("Version 2 file:", os.path.exists(v2_path))
print("Version 2 location:", v2_path)

print("\nReady to upgrade Version 2!")

Version 2 file: True
Version 2 location: /content/AI_Escape_Room/version_2/app.py

Ready to upgrade Version 2!


In [101]:
import shutil
import os

source = "/content/AI_Escape_Room/version_2/app.py"
backup = "/content/AI_Escape_Room/version_2/app_backup.py"

shutil.copy(source, backup)

print("Backup created successfully!")
print("Original Version 2 preserved!")

Backup created successfully!
Original Version 2 preserved!


In [102]:
from google.colab import files
import shutil
import os

uploaded = files.upload()

source = "AI_Escape_Room_Version2.py"

destination = "/content/AI_Escape_Room/version_2/app.py"

shutil.copy(source, destination)

print("New Version 2 uploaded successfully!")

Saving AI_Escape_Room_Version2.py to AI_Escape_Room_Version2 (1).py
New Version 2 uploaded successfully!


In [103]:
import py_compile

app = "/content/AI_Escape_Room/version_2/app.py"

py_compile.compile(app, doraise=True)

print("Version 2 syntax check: SUCCESS!")

Version 2 syntax check: SUCCESS!


Launch the upgraded application

In [104]:
import subprocess
import sys
import time
from google.colab import output

app = "/content/AI_Escape_Room/version_2/app.py"

log = open("/content/streamlit_v2.log", "w")

process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run",
        app,
        "--server.port", "8502",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(5)

print("Streamlit process:", process.poll())
print("AI Escape Room Version 2 launching!")

output.serve_kernel_port_as_window(8502)

Streamlit process: 1
AI Escape Room Version 2 launching!
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [105]:
import os
import subprocess

print("========== STREAMLIT LOGS ==========")

os.system("tail -60 /content/streamlit_v2.log")

print("\n========== RUNNING PROCESS ==========")

os.system("ps aux | grep streamlit")

========== STREAMLIT LOGS ==========

========== RUNNING PROCESS ==========


0

In [59]:
import os

print("Checking application files...")

paths = [
    "/content/AI_Escape_Room/version_2/app.py",
    "/content/AI_Escape_Room_Version2.py",
    "/content/AI_Escape_Room/Version2.py"
]

for path in paths:
    print(path, "EXISTS:", os.path.exists(path))

Checking application files...
/content/AI_Escape_Room/version_2/app.py EXISTS: True
/content/AI_Escape_Room_Version2.py EXISTS: True
/content/AI_Escape_Room/Version2.py EXISTS: False


In [60]:
import os
import time
import subprocess

# Start Version 2
app_path = "/content/AI_Escape_Room_Version2.py"
log_path = "/content/streamlit_v2.log"

# Stop any old Streamlit process on port 8502
os.system("pkill -f 'streamlit run.*8502' || true")

# Start the new application
cmd = f"streamlit run {app_path} --server.port 8502 --server.address 0.0.0.0"

with open(log_path, "w") as log:
    process = subprocess.Popen(
        cmd,
        shell=True,
        stdout=log,
        stderr=subprocess.STDOUT
    )

print("Starting AI Escape Room Version 2...")

time.sleep(10)

print("\nSERVER STATUS:")
print("Process running:", process.poll() is None)

print("\nSTREAMLIT LOGS:")
os.system("tail -30 /content/streamlit_v2.log")

Starting AI Escape Room Version 2...

SERVER STATUS:
Process running: True

STREAMLIT LOGS:


0

In [61]:
import subprocess
import socket
import os

print("1. STREAMLIT PROCESS")
os.system("ps aux | grep '[s]treamlit'")

print("\n2. PORT 8502 STATUS")

sock = socket.socket()
result = sock.connect_ex(("127.0.0.1", 8502))
sock.close()

print("Port 8502 OPEN:", result == 0)

print("\n3. STREAMLIT LOG FILE")

os.system("cat /content/streamlit_v2.log")

print("\n4. APPLICATION FILE")

print(os.path.exists("/content/AI_Escape_Room_Version2.py"))

1. STREAMLIT PROCESS

2. PORT 8502 STATUS
Port 8502 OPEN: True

3. STREAMLIT LOG FILE

4. APPLICATION FILE
True


In [62]:
import requests
import subprocess

url = "http://127.0.0.1:8502"

print("1. CHECKING STREAMLIT HEALTH")

try:
    r = requests.get(url + "/_stcore/health", timeout=15)
    print("Status:", r.status_code)
    print("Response:", r.text)
except Exception as e:
    print("Health check error:", e)

print("\n2. CHECKING WEBSITE")

try:
    r = requests.get(url, timeout=20)
    print("Website status:", r.status_code)
    print("HTML received:", len(r.text), "characters")
except Exception as e:
    print("Website error:", e)

print("\n3. PROCESS DETAILS")

result = subprocess.run(
    ["pgrep", "-af", "streamlit"],
    capture_output=True,
    text=True
)

print(result.stdout or "No Streamlit process found")

1. CHECKING STREAMLIT HEALTH
Status: 200
Response: ok

2. CHECKING WEBSITE
Website status: 200
HTML received: 7863 characters

3. PROCESS DETAILS
37976 /bin/sh -c streamlit run /content/AI_Escape_Room_Version2.py --server.port 8502 --server.address 0.0.0.0
37977 /usr/bin/python3 /usr/local/bin/streamlit run /content/AI_Escape_Room_Version2.py --server.port 8502 --server.address 0.0.0.0



In [143]:
import subprocess
import time
import requests

app_path = "/content/AI_Escape_Room/version_2/app.py"
log_path = "/content/streamlit_v2_debug.log"

# Start Streamlit with Colab-compatible settings
log = open(log_path, "w")

process = subprocess.Popen(
    [
        "streamlit", "run", app_path,
        "--server.port", "8503",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(10)

print("Process running:", process.poll() is None)

try:
    response = requests.get(
        "http://127.0.0.1:8503/_stcore/health",
        timeout=15
    )
    print("Health:", response.status_code)
    print("Response:", response.text)
except Exception as e:
    print("Error:", e)

print("\nStreamlit logs:")
with open(log_path) as f:
    print(f.read()[-4000:])

# Generate Colab browser link
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8503)")
print("\nNEW VERSION 2 WEBSITE:")
print(url)

Process running: True
Health: 200
Response: ok

Streamlit logs:


2026-11-14 20:58:33.758 Uvicorn server started on 0.0.0.0:8503

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8503
  Network URL: http://172.28.0.12:8503
  External URL: http://136.85.164.246:8503



NEW VERSION 2 WEBSITE:
https://8503-m-s-usw4a0-3havjy49mt55e-a.us-west4.prod.r.colab.dev


cc

In [80]:
!pip install -q -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.3.0 which is incompatible.


In [81]:
from google import genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")

try:
    client = genai.Client(api_key=api_key)

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents="Hello Gemini! Reply with: AI integration is working."
    )

    print("SUCCESS!")
    print(response.text)

except Exception as e:
    print("GEMINI ERROR:")
    print(type(e).__name__)
    print(str(e))

Enter your Gemini API key: ··········
SUCCESS!
AI integration is working.


In [82]:
!pip install -q -U google-genai

In [84]:
from google import genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=api_key)

# Using the verified active model gemini-3.5-flash-lite
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply exactly: Gemini is working!"
)

print(response.text)

Enter your Gemini API key: ··········
Gemini is working!


In [110]:
!fuser -k 8503/tcp || true

8503/tcp:            39155


new

In [112]:
import shutil

shutil.copy(
    "/content/AI_Escape_Room_Version2_Improved.py",
    "/content/AI_Escape_Room/version_2/app.py"
)

print("Version 2 updated successfully!")

Version 2 updated successfully!


In [113]:
!streamlit run /content/AI_Escape_Room/version_2/app.py --server.port 8503 --server.headless true > /content/streamlit_v2.log 2>&1 &

In [114]:
import requests

response = requests.get("http://localhost:8503/_stcore/health")
print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: ok


In [115]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8503)")
print(url)

https://8503-m-s-usw4a0-3havjy49mt55e-a.us-west4.prod.r.colab.dev


In [116]:
import shutil

shutil.copy(
    "/content/AI_Escape_Room_Version2_Timer_Fix.py",
    "/content/AI_Escape_Room/version_2/app.py"
)

print("Timer fix applied successfully!")

Timer fix applied successfully!


In [117]:
!fuser -k 8503/tcp || true

8503/tcp:            104211


In [118]:
!streamlit run /content/AI_Escape_Room/version_2/app.py --server.port 8503 --server.headless true > /content/streamlit_v2.log 2>&1 &

In [119]:
import requests

response = requests.get("http://localhost:8503/_stcore/health")
print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: ok


In [122]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8503)")
print(url)

https://8503-m-s-usw4a0-3havjy49mt55e-a.us-west4.prod.r.colab.dev
